# CALM



## Set up

In [1]:

from __future__ import annotations

import os
import sys
from pathlib import Path


def discover_calm_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "calm").is_dir():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy CALM: mở notebook từ thư mục chứa pyproject.toml hoặc `cd` vào repo rồi chạy lại."
    )


CALM_DIR = discover_calm_dir()
CALM_SRC = CALM_DIR / "src"
if str(CALM_SRC) not in sys.path:
    sys.path.insert(0, str(CALM_SRC))

from calm.utils.env_loader import load_env

load_env(CALM_DIR / ".env")
load_env(CALM_DIR.parent / ".env")

print("CALM_DIR =", CALM_DIR)
print("OPENAI_API_KEY:", "set" if os.environ.get("OPENAI_API_KEY") else "missing")
print("OPENROUTER_API_KEY:", "set" if os.environ.get("OPENROUTER_API_KEY") else "missing")


CALM_DIR = /home/hungvu/code/khanh/v2/calm-
OPENAI_API_KEY: missing
OPENROUTER_API_KEY: set


## Gọi LLM

In [2]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="arcee-ai/trinity-large-preview:free",
    max_completion_tokens=10000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Orchestrator factory

In [3]:
from calm.memory.chroma_store import ChromaMemoryStore
from calm.orchestrator import CALMOrchestrator

persist = str(CALM_DIR / ".chroma_test_notebook")
memory_store = ChromaMemoryStore(
    collection_name="calm_test_nb",
    persist_directory=persist,
    use_openai_embeddings=bool(os.environ.get("OPENAI_API_KEY")),
)

config = {
    "planner_n_max": 2,
    "qa_n_max": 2,
    "rsen_k": 2,
}

orch = CALMOrchestrator.from_llm(
    llm=llm,
    memory_store=memory_store,
    tools={},
    config=config,
)

assert orch.planner is not None
assert orch.router_agent is not None
assert orch.data_agent is not None
assert orch.qa_agent is not None
assert orch.prediction_agent is not None
assert orch.rsen is not None
assert orch.executor is not None
assert orch.memory_agent is not None

print("CALMOrchestrator.from_llm OK — đủ agent.")


[Orchestrator] No model_runner provided and no prediction checkpoint found. PredictionAgent may fall back to heuristic mode.


CALMOrchestrator.from_llm OK — đủ agent.


## QueryNormalizerAgent

Chuẩn hoá query thành context JSON


In [4]:
from calm.agents.query_normalizer_agent import QueryNormalizerAgent
from calm.tools.geocoding import GeocodingTool
from calm.tools.safety_check import SafetyChecker

safety_checker = SafetyChecker(llm=llm)
geocoder = GeocodingTool(safety_checker=safety_checker, config={})

normalizer = QueryNormalizerAgent(
    geocoder=geocoder,
    llm=llm,
    timezone="Asia/Bangkok",
)

qctx = normalizer.normalize("Predict wildfire risk for Central Valley California next 7 days")

print("keys:", sorted(k for k in qctx.keys() if not k.startswith("_")))
print("location_text:", qctx.get("location_text"))
print("location_resolved:", qctx.get("location_resolved"))
print("time_expression:", qctx.get("time_expression"))
print("time_range:", qctx.get("time_range"))
print("prediction_target:", qctx.get("prediction_target"))
print("requested_output:", qctx.get("requested_output"))
print("ambiguities:", qctx.get("ambiguities"))

keys: ['ambiguities', 'location_kind', 'location_resolved', 'location_text', 'prediction_target', 'requested_output', 'task_type', 'time_expression', 'time_range']
location_text: Central Valley California
location_resolved: {'source': 'geocoder', 'raw_text': None, 'name': 'Central Valley', 'lat': 38.0, 'lon': -121.4, 'bbox': {'min_lat': 37.99995, 'max_lat': 38.00005, 'min_lon': -121.40005, 'max_lon': -121.39995}, 'country': 'United States', 'admin1': 'California', 'admin2': 'San Joaquin County', 'resolution_kind': 'region_centroid_fallback', 'needs_region_bbox': True}
time_expression: next 7 days
time_range: {'start': '2026-03-25T10:56:14.186134+07:00', 'end': '2026-04-01T10:56:14.186134+07:00', 'timezone': 'Asia/Bangkok', 'source': 'local_parser'}
prediction_target: wildfire_risk
requested_output: ['risk_level']
ambiguities: [{'field': 'location', 'reason': 'region_resolved_as_point', 'message': 'Location appears to describe a large region, but geocoder returned a point-sized result o

## PlanningAgent



In [5]:
plan_state = orch.planner.invoke("What environmental factors affect Amazon wildfires?")
plan_steps = plan_state.get("final_output") or []
print("N steps:", len(plan_steps))
if plan_steps:
    print("First step keys:", list(plan_steps[0].keys()))
    print("First agent/action:", plan_steps[0].get("agent"), "|", plan_steps[0].get("action")[:60] if plan_steps[0].get("action") else "")
if plan_state.get("error"):
    print("Planner warning/error:", plan_state.get("error"))


N steps: 3
First step keys: ['step_id', 'action', 'agent', 'prompt', 'parameters', 'expected_output', 'success_criteria']
First agent/action: data_knowledge | retrieve_knowledge


## RouterAgent


In [6]:
query_demo = "Forecast wildfire likelihood in Northern California over the next 14 days"
routing = orch.router_agent.route(query_demo, plan_steps)
print("task_type:", routing.task_type)
print("confidence:", routing.confidence)
print("artifacts:", routing.required_artifacts[:5])
print("reasoning:", routing.reasoning[:120] if routing.reasoning else "")


task_type: prediction
confidence: 0.95
artifacts: ['prediction', 'met_data', 'spatial_data']
reasoning: Query explicitly requests a forecast/likelihood prediction over a specific time period


## DataKnowledgeAgent



In [7]:
from calm.utils.time_utils import resolve_time_range

params = {
    "location": "California, USA",
    "time_range": resolve_time_range("next 7 days", default_today=True),
}

data_res = orch.data_agent.retrieve("Wildfire conditions California", params)
print("Keys:", list(data_res.keys())[:10])
print("n retrieved:", len(data_res.get("retrieved_data") or []))
if data_res.get("error"):
    print("error:", data_res["error"][:200])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8408.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
GEE initialization failed: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.
Copernicus collection failed: [UNSAFE] Safety check blocked: Copernicus fetch_era5 lat=36.7014631 lon=-118.755997 time_range={'start': '2026-03-25', 'end': '2026-03-31', 'granularity': 'day', 'horizon_days': 7, 'is_forecast_window': False}


Keys: ['task_type', 'retrieval_summary', 'normalized_query', 'normalized_query_context', 'sub_requests', 'retrieved_data', 'model_inputs', 'data_quality', 'extracted_knowledge', 'reflection_trace']
n retrieved: 14
error: copernicus: [UNSAFE] Safety check blocked: Copernicus fetch_era5 lat=36.7014631 lon=-118.755997 time_range={'start': '2026-03-25', 'end': '2026-03-31', 'granularity': 'day', 'horizon_days': 7, 'is_for


## PredictionReasoningAgent — `predict`



In [8]:
from calm.utils.time_utils import resolve_time_range

pred_params = {
    "location": "California Central Valley",
    "latitude": 36.5,
    "longitude": -119.5,
    "time_range": resolve_time_range("next 7 days", default_today=True),
    "met_data": {
        "temperature": 28.0,
        "humidity": 40.0,
        "wind_speed": 12.0,
        "precipitation": 0.0,
    },
    "spatial_data": {"ndvi_mean": 0.35},
}

pred_out = orch.prediction_agent.predict(pred_params)
print("risk_level:", pred_out.get("risk_level"))
print("confidence:", pred_out.get("confidence"))
print("used_fallback:", pred_out.get("used_fallback"))
if pred_out.get("error"):
    print("error:", pred_out["error"][:200])


risk_level: Moderate
confidence: 0.65
used_fallback: True


## RSENModule — `validate`


In [9]:
val = orch.rsen.validate(
    prediction={"risk_level": pred_out.get("risk_level", "Medium"), "confidence": pred_out.get("confidence", 0.5)},
    met_data=pred_params["met_data"],
    spatial_data=pred_params["spatial_data"],
)
print("validation_decision:", val.get("validation_decision", val.get("decision")))
print("rationale (trunc):", (val.get("final_rationale") or "")[:200])


validation_decision: Plausible
rationale (trunc): The Final Prediction of 'Moderate' risk is supported by consistent findings from both specialist analyses. The Weather Analyst reports moderate temperatures, elevated humidity, and moderate wind speed


## WildfireQAAgent — `invoke`



In [10]:
qa_res = orch.qa_agent.invoke(
    "List two main natural causes of wildfires.",
    pre_retrieved=data_res if data_res.get("retrieved_data") else {"retrieved_data": []},
)
fo = qa_res.get("final_output") or {}
print("answer (trunc):", (fo.get("answer") or "")[:300])
print("citations:", len(fo.get("citations") or []))
print("approved:", qa_res.get("approved"))


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 5 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774483200000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}

## ExecutionAgent — `execute_step`


In [ ]:
step = {
    "step_id": "s1",
    "agent": "data_knowledge",
    "action": "retrieve environmental data",
    "parameters": {"location": "Amazon basin"},
    "prompt": "Wildfire drivers Amazon",
}

ctx = {
    "query": "Wildfire drivers Amazon",
    "original_query": "Wildfire drivers Amazon",
    "parameters": {"location": "Amazon basin"},
}

ex_res = orch.executor.execute_step(step, ctx)
print("executor keys:", list(ex_res.keys())[:8])
print("retrieved len:", len((ex_res.get("retrieved_data") or [])))


##  FeedbackRefinerAgent


In [ ]:
from calm.agents.feedback_refiner_agent import FeedbackRefinerAgent

refiner = FeedbackRefinerAgent(memory_store=memory_store, config={})
refined = refiner.run(
    prediction_output=pred_out,
    rsen_outputs=val,
    data_quality={"level": "medium", "sources_count": 1},
)
print("calibrated_confidence:", refined.get("calibrated_confidence"))
print("final_decision:", refined.get("final_decision"))
print("actions:", refined.get("refinement_actions", [])[:3])


## EvaluatorAgent — LLM-as-a-Judge



In [11]:
from calm.agents.evaluator_agent import EvaluatorAgent

evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})
orch_snapshot = orch.run("What is fuel moisture in wildfire context?")
ev = evaluator.evaluate(response=orch_snapshot, query="What is fuel moisture in wildfire context?")
print("average_score:", ev.get("average_score"))
print("passed:", ev.get("passed"))
print("scores:", ev.get("scores"))


Planning generator failed: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 5 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774483200000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}
Traceback (most recent call last):
  File "/home/hungvu/code/khanh/v2/calm-/src/calm/agents/planning_agent.py", line 168, in _generator_node
    resp = self.llm.invoke(msgs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 402, in invoke
    self.generate_prompt(
  File "/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/langchain_core/language_models/chat_models.py", line 1123, in generate_prompt
    return self.generate(prompt_messages, stop=stop, callbacks=callbacks, **kwargs)
           ^^^^^^^^^^^^^^^^^

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-min. ', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '16', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1774411500000'}, 'provider_name': None}}, 'user_id': 'user_38eFibBaOsM8LFJwMbOM2dAYpKh'}

## Intent hints (routing fallback)


In [ ]:
from calm.utils.intent_hints import infer_task_from_keywords

assert infer_task_from_keywords("What are risks of fire?") == "qa"
assert infer_task_from_keywords("Predict wildfire risk next 7 days") == "prediction"
print("intent_hints OK")


## SafetyChecker




In [ ]:
from calm.tools.safety_check import SafetyChecker

safety = SafetyChecker(llm=llm)
ok = safety.is_safe("Retrieve ERA5 summary for a bounding box in California for research.")
print("safe (benign action):", ok)


## CALMOrchestrator — luồng hoàn chỉnh


In [ ]:
def print_orch_result(label: str, r: dict) -> None:
    tt = r.get("task_type")
    print(f"\n=== {label} ===")
    print("task_type:", tt)
    print("error:", r.get("error"))
    if tt == "qa":
        print("answer (trunc):", (r.get("answer") or "")[:280])
    else:
        print("risk_level:", r.get("risk_level"), "| decision:", r.get("decision"))
        print("rationale (trunc):", (r.get("rationale") or "")[:200])
    mem = r.get("memory_reflection") or {}
    print("memory stored:", mem.get("stored"), mem.get("reason") or mem.get("mode") or "")




In [ ]:
r_qa = orch.run("What are common human causes of wildfires?")
print_orch_result("QA pipeline", r_qa)


In [ ]:
r_pr = orch.run("Predict wildfire risk for California Central Valley over the next 7 days")
print_orch_result("Prediction pipeline", r_pr)